# Noisy Adaptive Encoder Analysis

Use this notebook after running `scripts/adaptive_noise_sweep.py`, or for a single adaptive run whose config has `[train_input_noise]` enabled. It focuses on how noise scale/probability changes the selected encoder dimension, Stage 1/2 metrics, symbolic recovery files when present, and the stress-space sample distribution.

In [ ]:
from pathlib import Path
import json
import os
import sys
from pprint import pprint

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists() and (PROJECT_ROOT.parent / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))

MPLCONFIGDIR = PROJECT_ROOT / ".matplotlib-cache"
MPLCONFIGDIR.mkdir(exist_ok=True)
os.environ.setdefault("MPLCONFIGDIR", str(MPLCONFIGDIR))

CONFIG_PATH = PROJECT_ROOT / "configs" / "adaptive_encoder_rotated_hill.toml"
BASE_RUN_DIR = PROJECT_ROOT / "results" / "adaptive_rotatedhill_exp02"
NOISE_SWEEP_SUMMARY = BASE_RUN_DIR / "noise_sweep" / "noise_sweep_summary.json"

# For drilling into a specific row. Keep None to auto-pick the largest available scale.
SELECTED_SCALE = None
SELECTED_PROBABILITY = None
SELECTED_SEED = None

print("project:", PROJECT_ROOT)
print("config:", CONFIG_PATH)
print("noise sweep summary:", NOISE_SWEEP_SUMMARY)

In [ ]:
import numpy as np

try:
    get_ipython().run_line_magic("matplotlib", "inline")
except Exception:
    pass

import matplotlib.pyplot as plt

from invariant_generator.config import load_config
from invariant_generator.data import canonicalize_stress_features, load_hdf_dataset, split_surface_data
from invariant_generator.stress_viz import plot_stress_space_summary, simulate_train_input_noise

config = load_config(CONFIG_PATH)

def load_json(path):
    path = Path(path)
    if not path.exists():
        return None
    with path.open("r", encoding="utf-8") as f:
        return json.load(f)

def metric_from(row, stage, metric="rmse"):
    metrics = row.get(f"{stage}_test_metrics") or {}
    return metrics.get(metric)

def train_metric_from(row, stage, metric="rmse"):
    metrics = row.get(f"{stage}_train_metrics") or {}
    return metrics.get(metric)

def row_label(row):
    return f"scale={row.get('scale'):g}, p={row.get('probability'):g}, seed={row.get('seed')}"

## Noise Sweep Summary

In [ ]:
summary = load_json(NOISE_SWEEP_SUMMARY)
if summary is None:
    candidates = sorted((PROJECT_ROOT / "results").glob("*/noise_sweep/noise_sweep_summary.json"))
    if candidates:
        NOISE_SWEEP_SUMMARY = candidates[-1]
        summary = load_json(NOISE_SWEEP_SUMMARY)
        print("Using discovered sweep summary:", NOISE_SWEEP_SUMMARY)

rows = [] if summary is None else summary.get("rows", [])
if not rows:
    print("No noise sweep summary found. You can still use the stress-space EDA section for a single config/run.")
else:
    print("summary:", NOISE_SWEEP_SUMMARY)
    print("rows:", len(rows))
    for row in rows:
        print(
            row_label(row),
            "selected_n=", row.get("selected_n"),
            "stage1_test_rmse=", metric_from(row, "stage1_selected"),
            "stage2_test_rmse=", metric_from(row, "stage2"),
            "symbolic_test_rmse=", (row.get("symbolic_test_metrics") or {}).get("rmse"),
        )

## Noise vs Accuracy

In [ ]:
if rows:
    groups = sorted({(row.get("probability"), row.get("seed")) for row in rows})
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for probability, seed in groups:
        group = sorted(
            [row for row in rows if row.get("probability") == probability and row.get("seed") == seed],
            key=lambda row: row.get("scale", 0.0),
        )
        x = np.array([row.get("scale", np.nan) for row in group], dtype=float)
        stage1 = np.array([metric_from(row, "stage1_selected") for row in group], dtype=float)
        stage2 = np.array([metric_from(row, "stage2") for row in group], dtype=float)
        symbolic = np.array([(row.get("symbolic_test_metrics") or {}).get("rmse", np.nan) for row in group], dtype=float)
        label = f"p={probability:g}, seed={seed}"
        axes[0].plot(x, stage1, marker="o", label=label)
        if np.isfinite(stage2).any():
            axes[1].plot(x, stage2, marker="o", label=label)
        if np.isfinite(symbolic).any():
            axes[2].plot(x, symbolic, marker="o", label=label)

    for ax, title in zip(axes, ["Stage 1 selected test RMSE", "Stage 2 test RMSE", "Stage 3 symbolic test RMSE"]):
        ax.set_xlabel("train input noise scale")
        ax.set_ylabel("RMSE")
        ax.set_title(title)
        ax.set_yscale("log")
        ax.legend()
    fig.tight_layout()
    plt.show()

    fig, ax = plt.subplots(figsize=(7, 4))
    for probability, seed in groups:
        group = sorted(
            [row for row in rows if row.get("probability") == probability and row.get("seed") == seed],
            key=lambda row: row.get("scale", 0.0),
        )
        x = np.array([row.get("scale", np.nan) for row in group], dtype=float)
        n_values = np.array([row.get("selected_n", np.nan) for row in group], dtype=float)
        ax.plot(x, n_values, marker="o", label=f"p={probability:g}, seed={seed}")
    ax.set_xlabel("train input noise scale")
    ax.set_ylabel("selected encoder dimension n")
    ax.set_title("Adaptive selected n vs noise")
    ax.legend()
    fig.tight_layout()
    plt.show()

## Selected Noise Case Details

In [ ]:
selected_row = None
if rows:
    candidates = rows
    if SELECTED_SCALE is not None:
        candidates = [row for row in candidates if np.isclose(row.get("scale", np.nan), SELECTED_SCALE)]
    if SELECTED_PROBABILITY is not None:
        candidates = [row for row in candidates if np.isclose(row.get("probability", np.nan), SELECTED_PROBABILITY)]
    if SELECTED_SEED is not None:
        candidates = [row for row in candidates if row.get("seed") == SELECTED_SEED]
    if candidates:
        selected_row = sorted(candidates, key=lambda row: row.get("scale", -np.inf))[-1]
        print("selected:", row_label(selected_row))
        pprint(selected_row)
    else:
        print("No row matched the selected filters.")

In [ ]:
if selected_row is not None:
    stage1 = load_json(selected_row.get("stage1_summary"))
    stage2 = load_json(selected_row.get("stage2_summary"))

    if stage1 is not None:
        runs = stage1.get("runs", [])
        metric = stage1.get("metric", "rmse")
        n_values = np.array([row["n"] for row in runs], dtype=int)
        train_values = np.array([row["train_metrics"].get(metric, np.nan) for row in runs], dtype=float)
        test_values = np.array([row["test_metrics"].get(metric, np.nan) for row in runs], dtype=float)
        fig, ax = plt.subplots(figsize=(7, 4))
        ax.plot(n_values, train_values, marker="o", label=f"train {metric}")
        ax.plot(n_values, test_values, marker="o", label=f"test {metric}")
        if stage1.get("selected_n") is not None:
            ax.axvline(stage1["selected_n"], color="black", linestyle="--", linewidth=1, label="selected n")
        ax.set_xlabel("encoder output dimension n")
        ax.set_ylabel(metric)
        ax.set_yscale("log")
        ax.set_title("Stage 1 adaptive sweep for selected noise case")
        ax.legend()
        fig.tight_layout()
        plt.show()

    if stage2 is not None:
        def load_stage2_history(path_key, inline_key):
            path = stage2.get(path_key)
            if path:
                payload = load_json(path)
                if payload is not None:
                    return payload.get("history", [])
            return stage2.get(inline_key, [])

        histories = [
            (load_stage2_history("sparse_history_path", "sparse_history"), "sparsity"),
            (load_stage2_history("refit_history_path", "refit_history"), "masked refit"),
        ]
        fig, axes = plt.subplots(1, 2, figsize=(12, 4))
        plotted = False
        for history, label in histories:
            if not history:
                continue
            plotted = True
            epochs = np.array([item["epoch"] for item in history], dtype=float)
            for key, ax in [("loss_total", axes[0]), ("loss_data", axes[1])]:
                values = np.array([item.get(key, np.nan) for item in history], dtype=float)
                ax.plot(epochs, values, label=label)
                ax.set_xlabel("epoch")
                ax.set_ylabel(key)
                ax.set_yscale("log")
        if plotted:
            axes[0].set_title("Stage 2 total objective")
            axes[1].set_title("Stage 2 data loss")
            for ax in axes:
                ax.legend()
            fig.tight_layout()
            plt.show()
        else:
            plt.close(fig)
            print("No Stage 2 histories found for selected row.")

        formulas = stage2.get("formulas", {}).get("formulas", [])
        if formulas:
            print("Final encoded invariant formulas:")
            for item in formulas:
                print(item.get("raw_formula"))
                print("  active:", ", ".join(item.get("active_terms", [])) or "none")

    if selected_row.get("best_equation"):
        print("
Best symbolic equation:")
        print(selected_row["best_equation"])
else:
    print("Select a noise-sweep row above to inspect per-run histories.")

## Stress-Space Data EDA

In [ ]:
X_raw = load_hdf_dataset(
    config.data.data_dir,
    config.data.dataset_name,
    dataset_key=config.data.dataset_key,
)
X_clean, _ = canonicalize_stress_features(
    X_raw,
    stress_format=config.data.stress_format,
)
X_train_raw, X_test_raw, _ = split_surface_data(
    X_clean,
    test_size=config.data.test_size,
    random_state=config.data.random_state,
    shuffle=config.data.shuffle,
    split_path=config.split_path,
    load_if_exists=config.train.use_saved_split,
    save_if_missing=config.train.save_split_if_missing,
    surface_target=config.augmentation.surface_target,
)
print("raw train surface:", X_train_raw.shape)
print("raw test surface:", X_test_raw.shape)

if selected_row is not None:
    scale = float(selected_row.get("scale", 0.0))
    probability = float(selected_row.get("probability", 1.0))
    seed = int(selected_row.get("seed", config.train_input_noise.random_state))
    relative = bool(selected_row.get("train_input_noise", {}).get("relative_to_feature_std", True))
elif config.train_input_noise.enabled:
    scale = config.train_input_noise.scale
    probability = config.train_input_noise.probability
    seed = config.train_input_noise.random_state
    relative = config.train_input_noise.relative_to_feature_std
else:
    scale = 0.0
    probability = 0.0
    seed = config.data.random_state
    relative = True

if scale > 0.0 and probability > 0.0:
    X_noisy_view = simulate_train_input_noise(
        X_train_raw,
        scale=scale,
        probability=probability,
        seed=seed,
        relative_to_feature_std=relative,
    )
    title = f"Raw train surface with simulated train-time noise (scale={scale:g}, p={probability:g})"
    fig = plot_stress_space_summary(
        X_train_raw,
        noisy=X_noisy_view,
        title=title,
        max_samples=5000,
        seed=seed,
    )
else:
    fig = plot_stress_space_summary(
        X_train_raw,
        title="Raw clean train surface",
        max_samples=5000,
        seed=seed,
    )
plt.show()

## Notes for Plot Interpretation

The PCA view is a generic 2D projection of the 6D stress vectors. The principal-stress and deviatoric-plane views are more physical projections: they show how the surface looks after diagonalizing each stress tensor and after removing hydrostatic pressure. The noisy overlay is a simulated snapshot of the same train-time noise policy; during training, noise is resampled every batch/epoch.